# Fundamentos Matemáticos para IA - Aula 01
## Convolução, Bias e ReLU em Redes Neurais Convolucionais

**PPGIA - IFGoiano Campus Urutaí** | Prof. Julio Cesar Ferreira | 19/03/2026

---

## Sobre o Curso

Este curso de **40 horas** foca em compreender os fundamentos matemáticos por trás dos modelos de Inteligência Artificial, organizado em **4 pilares**:

| Pilar | O que estuda | Para que serve na IA |
|-------|-------------|---------------------|
| **Álgebra Linear** | Vetores, matrizes, transformações | Representar e transformar dados |
| **Cálculo** | Gradiente, derivadas | Treinar modelos (backpropagation) |
| **Probabilidade** | Distribuições, estatística | Predições e incertezas |
| **Otimização** | Função de perda, métricas | Ajustar parâmetros do modelo |

**Metodologia:** Ao invés de começar com definições abstratas, partimos de um **modelo prático (CNN)** e extraímos os conceitos matemáticos por trás dele.

---
## O que é uma CNN? (Visão Geral)

Uma **CNN (Rede Neural Convolucional)** é um modelo de IA que recebe um dado de entrada (no nosso caso, uma imagem) e produz uma **predição** na saída.

```
 ENTRADA                    PROCESSAMENTO                          SAÍDA
┌──────────┐     ┌──────────────┐   ┌──────────────┐    ┌─────────────┐
│  Imagem  │ →→→ │  Camadas de   │ → │   Camadas    │ →→ │  Predição   │
│  64x64   │     │  Convolução   │   │   Densas     │    │ P(óculos)  │
│ 3 canais │     │ +Bias +ReLU  │   │ (classif.) │    │ P(sorrindo)│
│  (RGB)   │     │              │   │            │    │ P(jovem)   │
└──────────┘     └──────────────┘   └──────────────┘    └─────────────┘
```

**Nesta aula**, focamos nas **Camadas de Convolução** — a primeira etapa do processamento.

> **Analogia:** Pense na CNN como uma fábrica. A imagem entra como matéria-prima. Em cada camada (estação da fábrica), ela é transformada: primeiro detectamos bordas, depois formas, depois objetos. No final, a fábrica entrega o resultado: "esta pessoa está sorrindo com 92% de certeza".

---
## Entendendo Imagens Digitais

Antes de falar de convolução, precisamos entender como o computador "vê" uma imagem.

### O que é um Pixel?
Um **pixel** é o menor ponto de uma imagem digital. Cada pixel tem um **valor numérico** que representa sua intensidade (brilho):
- **0 = preto** (ausência de luz)
- **255 = branco** (máxima luz)
- Valores intermediários = tons de cinza

### O que é uma Matriz?
Uma **matriz** é simplesmente uma tabela de números organizada em linhas e colunas. Uma imagem em escala de cinza **é** uma matriz, onde cada célula contém o valor de um pixel.

### Canais de Cor (RGB)
Uma foto colorida tem **3 canais**: Vermelho (R), Verde (G) e Azul (B). Cada canal é uma matriz separada. A cor final de cada pixel é a combinação dos 3 canais.

```
Imagem Colorida = 3 matrizes empilhadas

  Canal R          Canal G          Canal B
┌─────────┐    ┌─────────┐    ┌─────────┐
│ 200 150 │    │  50  30 │    │  20  10 │
│ 180 160 │    │  40  35 │    │  15  12 │
└─────────┘    └─────────┘    └─────────┘
```

Nesta aula, trabalhamos com imagens de **1 canal só** (escala de cinza) para simplificar. Mas o conceito se aplica igualmente a imagens coloridas.

> **Na CNN real:** a cada camada de convolução, o número de canais **aumenta** (ex: 3 → 16 → 64 → 128). Cada canal extraí uma característica diferente (borda, textura, forma...).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter
import os

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
np.set_printoptions(precision=2, suppress=True)

print('Imports OK!')

---
## Carregando a Imagem da Aula

Vamos carregar a mesma imagem aérea usada pelo professor: uma foto de satélite em escala de cinza mostrando um carro estacionado, árvores, telhado e rua.

In [ ]:
# Caminho da imagem usada na aula
IMG_PATH = 'images/imagem_aerea_professor.png'

# Abrir a imagem e converter para escala de cinza ('L' = luminance = 1 canal só)
img_pil = Image.open(IMG_PATH).convert('L')

# Converter para array NumPy (matriz de números) e normalizar para 0.0-1.0
# Original: 0 a 255 (inteiros) -> Dividimos por 255 -> 0.0 a 1.0 (decimais)
img = np.array(img_pil).astype(float) / 255.0

print(f'Dimensões: {img.shape[0]} linhas x {img.shape[1]} colunas')
print(f'Total de pixels: {img.shape[0] * img.shape[1]:,}')
print(f'Valores: de {img.min():.3f} (escuro) até {img.max():.3f} (claro)')

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Imagem de Entrada ($Im_L$)', fontsize=14)
axes[0].set_xlabel('Colunas (pixels)')
axes[0].set_ylabel('Linhas (pixels)')
plt.colorbar(im, ax=axes[0], label='Intensidade (0=preto, 1=branco)')

# Mostrar uma região como números
regiao = img[100:108, 120:128]
axes[1].imshow(regiao, cmap='gray', vmin=0, vmax=1)
for i in range(regiao.shape[0]):
    for j in range(regiao.shape[1]):
        axes[1].text(j, i, f'{regiao[i,j]:.2f}', ha='center', va='center',
                     fontsize=8, color='red' if regiao[i,j] > 0.5 else 'yellow')
axes[1].set_title('Zoom: cada pixel \u00e9 um N\u00daMERO', fontsize=14)

plt.suptitle('Para o computador, uma imagem \u00e9 uma TABELA DE N\u00daMEROS (matriz)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nO que vemos: carro branco, para-brisa escuro, \u00e1rvore, telhado, rua.')
print('O que o computador v\u00ea: uma tabela com milhares de n\u00fameros entre 0 e 1.')

---
## O que é Convolução?

### Definição intuitiva
A convolução é uma operação de **somas e multiplicações** que transforma uma imagem usando um **filtro (kernel)**.

> **Analogia:** Imagine que você usa óculos com lentes especiais. Cada tipo de lente mostra a imagem de um jeito diferente: uma lente destaca bordas, outra borra a imagem, outra realce cores. O **kernel** é a lente, e a **convolução** é o ato de olhar através dela.

### Definição matemática
No contínuo, a convolução entre duas funções $f$ e $g$ é:

$$(f * g)(t) = \int_{-\infty}^{\infty} f(\tau) \cdot g(t - \tau) \, d\tau$$

Propriedades importantes: é **comutativa** ($f*g = g*f$), **associativa** e tem **elemento neutro** (delta de Dirac).

Na prática computacional (matemática discreta), não usamos integrais. Usamos **somas e multiplicações** entre a imagem e o kernel:

$$Im_{L+1}[i,j] = \sum_{(m,n) \in \omega} Im_L[i+m, j+n] \cdot K[m,n]$$

Onde:
- $Im_L$ = imagem de entrada (camada L)
- $K$ = kernel (filtro escolhido)
- $Im_{L+1}$ = imagem convoluída (resultado, camada seguinte)

### O que é um Kernel?
O kernel é uma **pequena matriz** (geralmente 3x3) com valores escolhidos de acordo com o que queremos detectar na imagem. Kernels diferentes extraem características diferentes:
- Bordas verticais, bordas horizontais
- Suavização (blur)
- Realce de detalhes (sharpen)
- Textura, olhos, boca... (em camadas mais profundas)

In [ ]:
# Funções auxiliares

def convolucao_manual(imagem, kernel, padding=0, stride=1):
    """
    Convolução 2D manual.
    Cada passo: desliza uma janela pela imagem e calcula o produto interno com o kernel.
    """
    if padding > 0:
        imagem = np.pad(imagem, pad_width=padding, mode='constant', constant_values=0)
    kh, kw = kernel.shape
    ih, iw = imagem.shape
    oh = (ih - kh) // stride + 1
    ow = (iw - kw) // stride + 1
    saida = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            ri, rj = i * stride, j * stride
            janela = imagem[ri:ri+kh, rj:rj+kw]
            saida[i, j] = np.sum(janela * kernel)
    return saida


def relu(x):
    """ReLU: max(0, x)"""
    return np.maximum(0, x)


print('Funções prontas!')

---
## Convolução na Imagem Real (Slide da Aula)

O professor aplicou na imagem aérea o kernel **detector de bordas verticais**:

```
K = [[-1, 0, 1],
     [-1, 0, 1],
     [-1, 0, 1]]
```

> **Por que esse kernel detecta bordas?** Observe os valores: a coluna esquerda tem -1 e a direita tem +1. Quando o kernel está sobre uma região uniforme (mesma cor), os -1 cancelam os +1 e o resultado é zero. Mas quando há **mudança de intensidade** (ex: capô branco ao lado de para-brisa escuro), os valores não se cancelam e o resultado é alto. Isso é o que chamamos de **alta frequência**.

In [ ]:
# Definir o kernel (filtro) detector de bordas verticais
# Coluna esquerda = -1 (penaliza), centro = 0 (ignora), direita = +1 (premia)
# Onde houver mudança de escuro->claro da esquerda p/ direita, o resultado será alto
K = np.array([[-1, 0, 1],
              [-1, 0, 1],
              [-1, 0, 1]], dtype=float)

# Aplicar convolução com padding=1
# padding=1 adiciona borda de zeros para que a saída tenha o MESMO tamanho da entrada
img_conv = convolucao_manual(img, K, padding=1)

# Visualizar igual ao slide do professor
fig = plt.figure(figsize=(18, 6))

ax1 = fig.add_subplot(1, 3, 1)
im1 = ax1.imshow(img, cmap='gray', vmin=0, vmax=1)
ax1.set_title('$Im_L$ (Entrada)', fontsize=15, fontweight='bold')
plt.colorbar(im1, ax=ax1, fraction=0.046)

ax2 = fig.add_subplot(1, 3, 2)
ax2.imshow(K, cmap='RdBu', vmin=-1, vmax=1)
for i in range(3):
    for j in range(3):
        ax2.text(j, i, f'{int(K[i,j]):+d}', ha='center', va='center',
                 fontsize=22, fontweight='bold')
ax2.set_title('Kernel (K)\nDetector de Bordas Verticais', fontsize=14)
ax2.set_xticks([]); ax2.set_yticks([])

ax3 = fig.add_subplot(1, 3, 3)
im3 = ax3.imshow(img_conv, cmap='gray')
ax3.set_title('$Im_{L+1}$ (Convolu\u00edda)', fontsize=15, fontweight='bold')
plt.colorbar(im3, ax=ax3, fraction=0.046)

plt.suptitle('Slide da Aula: Convolu\u00e7\u00e3o detecta mudan\u00e7as de intensidade',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('O que observar na imagem convolu\u00edda:')
print('  - Onde h\u00e1 mudan\u00e7a brusca de cor (claro/escuro), aparecem linhas claras ou escuras')
print('  - Cap\u00f4 branco ao lado do para-brisa escuro = BORDA FORTE')
print('  - \u00c1reas uniformes ficam cinza neutro (sem mudan\u00e7a = sem borda)')

---
## Exemplo 1 (Quadro): Convolução 3x3 com Kernel 3x3

> **O que é produto interno?** É quando pegamos dois conjuntos de números do mesmo tamanho, multiplicamos cada par correspondente e somamos tudo. Exemplo: `[1,2,3]` e `[4,5,6]` → `1×4 + 2×5 + 3×6 = 4+10+18 = 32`.

### O que o professor fez no quadro:

1. Pegou a imagem 3x3 e o kernel 3x3
2. **Achatou** (flatten) ambos em vetores (linhas de números)
3. Calculou o **produto interno** entre eles
4. Resultado: um único número = **6**

**Conclusão fundamental: cada passo da convolução é um PRODUTO INTERNO!**

In [ ]:
Im_ex1 = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])

K_ex1 = np.array([[-1, 0, 1],
                  [-1, 0, 1],
                  [-1, 0, 1]])

print('=== EXEMPLO 1 ===')
print(f'Im_L (imagem 3x3):\n{Im_ex1}\n')
print(f'K (kernel 3x3):\n{K_ex1}\n')

# Passo 1: Flatten (achatar)
im_flat = Im_ex1.flatten()  # [1,2,3,4,5,6,7,8,9]
k_flat = K_ex1.flatten()    # [-1,0,1,-1,0,1,-1,0,1]

print(f'Flatten da imagem: {im_flat}')
print(f'Flatten do kernel: {k_flat}\n')

# Passo 2: Produto interno (multiplicar par a par e somar)
print('Produto interno (par a par):')
termos = im_flat * k_flat
for a, b, t in zip(im_flat, k_flat, termos):
    print(f'  {a:2d} x {b:+2d} = {t:+3.0f}')
print(f'             \u2500\u2500\u2500\u2500')
print(f'   SOMA      = {int(termos.sum())}')

print(f'\n\u2705 Im_L+1 = {int(np.dot(im_flat, k_flat))}')
print('\nConclus\u00e3o: convolu\u00e7\u00e3o 3x3 com kernel 3x3 = UM PRODUTO INTERNO')

In [ ]:
# Visualização do Exemplo 1
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, mat, titulo, cmap_name in [(axes[0], Im_ex1, 'Im_L (3x3)', 'gray'),
                                    (axes[1], K_ex1, 'Kernel K (3x3)', 'RdBu')]:
    vmin, vmax = (0, 10) if cmap_name == 'gray' else (-1, 1)
    ax.imshow(mat, cmap=cmap_name, vmin=vmin, vmax=vmax)
    for i in range(3):
        for j in range(3):
            cor = 'red' if cmap_name == 'gray' else 'black'
            ax.text(j, i, str(mat[i,j]), ha='center', va='center',
                    fontsize=18, fontweight='bold', color=cor)
    ax.set_title(titulo, fontsize=13)

axes[2].imshow([[6]], cmap='viridis', vmin=0, vmax=10)
axes[2].text(0, 0, '6', ha='center', va='center',
             color='white', fontsize=30, fontweight='bold')
axes[2].set_title('Resultado = 6\n(1 n\u00famero!)', fontsize=13)

plt.suptitle('Exemplo 1: Im_L e K com mesmo tamanho \u2192 resultado \u00e9 1 n\u00famero (produto interno)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Exemplo 2 (Quadro): Imagem 4x4 com Kernel 3x3

E quando a imagem é **maior** que o kernel? Precisamos de **4 passos**:

> **Analogia da janela:** Imagine uma janela de vidro 3x3 que você desliza por cima de um mosaico 4x4. A cada posição, você anota o que vê. Ao deslizar da esquerda para a direita e de cima para baixo, você captura várias "fotos" parciais do mosaico.

| Passo | Nome | O que faz |
|-------|------|----------|
| P1 | **Sliding Windows** | Desliza janela 3x3 pela imagem, extraindo sub-imagens |
| P2 | **Flatten** | Achata cada janela em uma linha e empilha tudo em uma matriz |
| P3 | **Multiplicação** | Multiplica a matriz pelo kernel transposto (vários produtos internos) |
| P4 | **Reshape** | Reorganiza o resultado em forma de imagem |

In [ ]:
Im_ex2 = np.array([[ 1,  2,  3,  4],
                   [ 5,  6,  7,  8],
                   [ 9, 10, 11, 12],
                   [13, 14, 15, 16]])

print('=== EXEMPLO 2 ===')
print(f'Im_L (4x4):\n{Im_ex2}\n')

# P1: Sliding Windows
print('P1 - SLIDING WINDOWS (extraindo janelas 3x3):')
print('    Deslizamos a janela: esquerda\u2192direita, depois desce uma linha\n')

janelas = []
posicoes = [(0,0), (0,1), (1,0), (1,1)]
for i, j in posicoes:
    w = Im_ex2[i:i+3, j:j+3]
    janelas.append(w)
    print(f'  Janela {len(janelas)} [linha {i}, coluna {j}]: {w.flatten()}')
print(f'\n  Total: {len(janelas)} janelas  (f\u00f3rmula: (4-3+1)\u00b2 = 4)')

In [ ]:
# Visualizar as janelas deslizando
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
cores = ['red', 'blue', 'green', 'orange']

for idx, (pos, cor) in enumerate(zip(posicoes, cores)):
    axes[idx].imshow(Im_ex2, cmap='gray', vmin=0, vmax=17)
    for ii in range(4):
        for jj in range(4):
            axes[idx].text(jj, ii, str(Im_ex2[ii,jj]), ha='center', va='center',
                          fontsize=12, fontweight='bold', color='white')
    rect = plt.Rectangle((pos[1]-0.5, pos[0]-0.5), 3, 3,
                         linewidth=3, edgecolor=cor, facecolor='none')
    axes[idx].add_patch(rect)
    axes[idx].set_title(f'Janela {idx+1}', color=cor, fontweight='bold')

# Resultado
Im_L1_ex2 = convolucao_manual(Im_ex2.astype(float), K, padding=0)
axes[4].imshow(Im_L1_ex2, cmap='viridis')
for i in range(2):
    for j in range(2):
        axes[4].text(j, i, str(int(Im_L1_ex2[i,j])), ha='center', va='center',
                     color='white', fontsize=16, fontweight='bold')
axes[4].set_title('Sa\u00edda 2x2', fontsize=13, color='red')

plt.suptitle('P1: Sliding Windows \u2014 deslizando janela 3x3 pela imagem 4x4',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# P2: Flatten | P3: Multiplicação
print('P2 - FLATTEN (empilhar janelas em matriz):')
matriz = np.array([w.flatten() for w in janelas])
print(f'  Matriz de janelas (4 linhas x 9 colunas):\n{matriz}\n')

k_T = K_ex1.flatten().reshape(-1, 1)
print(f'  Kernel transposto (9x1): {k_T.flatten()}\n')

print('P3 - MULTIPLICA\u00c7\u00c3O (cada linha x kernel = produto interno):')
resultado = matriz @ k_T
for idx in range(4):
    pi = np.dot(matriz[idx], K_ex1.flatten())
    print(f'  Janela {idx+1}: produto interno = {int(pi)}')

print(f'\nP4 - RESHAPE: vetor {resultado.flatten()} \u2192 matriz 2x2')
print(f'\n\u26a0\ufe0f  Problema: entrada era 4x4, sa\u00edda ficou 2x2. A imagem ENCOLHEU!')
print('   Solu\u00e7\u00e3o: PADDING (pr\u00f3ximo exemplo)')

---
## Exemplo 3 (Quadro): Padding

> **O que é Padding?** É como colocar uma moldura preta ao redor de uma foto. Preenchemos a borda da imagem com **zeros** (preto) antes de fazer a convolução. Assim, o resultado mantém o **mesmo tamanho** da imagem original.

> **O que significa "alta frequência"?** Em processamento de imagens, "alta frequência" = **mudança rápida de intensidade** (ex: a transição brusca do branco para o preto). "Baixa frequência" = regiões suaves, uniformes. O kernel detector de bordas retorna valores ALTOS onde há alta frequência.

O professor usou uma imagem com contraste forte (1 na esquerda, 10 na direita).

O resultado mostra **27 no centro** = mudança de intensidade muito forte!

In [ ]:
Im_ex3 = np.array([[ 1.,  1., 10.],
                   [ 1.,  1., 10.],
                   [ 1.,  1., 10.]])

Im_padded = np.pad(Im_ex3, 1, mode='constant', constant_values=0)

print('=== EXEMPLO 3: PADDING ===')
print(f'Im_L original (3x3) \u2014 escura na esquerda, clara na direita:')
print(f'{Im_ex3.astype(int)}\n')

print(f'Com zero-padding (borda de zeros \u2192 5x5):')
print(f'{Im_padded.astype(int)}\n')

Im_L1_ex3 = convolucao_manual(Im_ex3, K, padding=1)
print(f'Im_L+1 (3x3) \u2014 MANTEVE a resolu\u00e7\u00e3o:')
print(f'{Im_L1_ex3.astype(int)}\n')

print(f'\u2705 Centro = {int(Im_L1_ex3[1,1])} \u2192 Alta frequ\u00eancia! (grande mudan\u00e7a de 1 para 10)')
print(f'   Bordas = valores menores (transi\u00e7\u00e3o mais suave, por causa dos zeros do padding)')

In [ ]:
# Visualização
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, mat, titulo, cmap_name in [
    (axes[0], Im_ex3, '1. Original (3x3)', 'gray'),
    (axes[1], Im_padded, '2. Com Padding (5x5)', 'gray'),
    (axes[2], K, '3. Kernel (3x3)', 'RdBu'),
    (axes[3], Im_L1_ex3, '4. Sa\u00edda (3x3)', 'hot'),
]:
    vmin = -1 if cmap_name == 'RdBu' else (-5 if cmap_name == 'hot' else 0)
    vmax = 1 if cmap_name == 'RdBu' else (30 if cmap_name == 'hot' else 10)
    ax.imshow(mat, cmap=cmap_name, vmin=vmin, vmax=vmax)
    h, w = mat.shape
    for i in range(h):
        for j in range(w):
            v = int(mat[i,j])
            cor = 'blue' if cmap_name == 'hot' else ('black' if cmap_name == 'RdBu' else ('gray' if v == 0 else 'red'))
            ax.text(j, i, str(v), ha='center', va='center',
                    fontsize=13 if h <= 3 else 11, fontweight='bold', color=cor)
    ax.set_title(titulo, fontsize=12)

plt.suptitle('Exemplo 3: Padding mant\u00e9m resolu\u00e7\u00e3o | Centro=27 = borda forte (alta frequ\u00eancia)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Stride (Deslocamento da Janela)

> **O que é stride?** É o tamanho do "passo" que a janela dá ao deslizar pela imagem. Stride=1 move 1 pixel por vez (mais detalhado). Stride=2 pula 2 pixels (menos detalhes, imagem menor).

**Fórmula da dimensão de saída:**

$$\text{sa\u00edda} = \frac{\text{entrada} - \text{kernel} + 2 \times \text{padding}}{\text{stride}} + 1$$

Na prática:
- **Stride=1**: mantém mais informação (usado nas primeiras camadas)
- **Stride>1**: reduz tamanho de propósito (quando queremos diminuir resolução)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img, cmap='gray')
axes[0].set_title(f'Entrada {img.shape[0]}x{img.shape[1]}', fontsize=13)

for idx, s in enumerate([1, 2, 4]):
    r = convolucao_manual(img, K, padding=1, stride=s)
    axes[idx+1].imshow(r, cmap='gray')
    formula = f'({img.shape[0]}-3+2)/{s}+1 = {r.shape[0]}'
    axes[idx+1].set_title(f'Stride={s} \u2192 {r.shape[0]}x{r.shape[1]}\n{formula}', fontsize=12)

plt.suptitle('Stride maior = passo maior = sa\u00edda menor (menos detalhes)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Bias (Viés)

> **O que é bias?** É um número que **somamos** a cada pixel da imagem após a convolução. Funciona como um "ajuste fino" — desloca todos os valores um pouco para cima ou para baixo.
>
> **Analogia:** Imagine o controle de brilho de uma TV. Girar para cima (bias positivo) clareia tudo. Girar para baixo (bias negativo) escurece tudo. Sozinho, parece pouca coisa. Mas combinado com a ReLU, faz toda a diferença.

A fórmula com bias:

$$Im_{L+1}[i,j] = \left(\sum_{(m,n)} Im_L[i+m, j+n] \cdot K[m,n]\right) + b$$

In [ ]:
conv_img = convolucao_manual(img, K, padding=1)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
biases = [0.0, -0.1, -0.2, -0.4]

for idx, b in enumerate(biases):
    r = conv_img + b
    im = axes[idx].imshow(r, cmap='gray')
    axes[idx].set_title(f'Conv + Bias = {b:+.1f}', fontsize=13)
    plt.colorbar(im, ax=axes[idx], fraction=0.046)

plt.suptitle('Bias desloca todos os valores \u2014 sozinho, muda pouco visualmente',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('O poder do bias aparece quando combinado com ReLU (pr\u00f3xima se\u00e7\u00e3o)!')

---
## Função de Ativação ReLU

$$ReLU(z) = \max(0, z)$$

> **O que a ReLU faz?** Regra simplíssima: se o valor é **negativo**, vira **zero**. Se é positivo, permanece igual. É como um filtro que só deixa passar o que é "relevante" (positivo) e descarta o resto.
>
> **Por que é importante?** Sem a ReLU, a CNN só faria operações lineares (somas e multiplicações). Com operações lineares, não é possível aprender padrões complexos como rostos ou objetos. A ReLU insere **não-linearidade**, que permite à rede aprender relações complexas.
>
> **Presente em quase todas as camadas** de uma CNN: convolução → bias → **ReLU** → próxima camada.

In [ ]:
# Gráfico da função ReLU
z = np.linspace(-3, 3, 200)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(z, relu(z), 'b-', linewidth=3)
axes[0].fill_between(z[z<0], relu(z[z<0]), alpha=0.3, color='red', label='Negativo \u2192 vira ZERO')
axes[0].fill_between(z[z>=0], 0, relu(z[z>=0]), alpha=0.2, color='green', label='Positivo \u2192 mant\u00e9m')
axes[0].axhline(0, color='k', linewidth=0.5)
axes[0].axvline(0, color='k', linewidth=0.5)
axes[0].set_title('Fun\u00e7\u00e3o ReLU: max(0, z)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Valor do pixel (ap\u00f3s conv + bias)')
axes[0].set_ylabel('Sa\u00edda da ReLU')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# Exemplo prático
valores = [-3, -1, 0, 0.5, 2, 5, -0.5, -2]
valores_relu = [max(0, v) for v in valores]
x = range(len(valores))
axes[1].bar(x, valores, color=['red' if v<0 else 'green' for v in valores], alpha=0.4, label='Antes')
axes[1].bar(x, valores_relu, width=0.4, color=['red' if v<0 else 'green' for v in valores], alpha=0.9, label='Depois')
for i, (v, vr) in enumerate(zip(valores, valores_relu)):
    axes[1].text(i, max(v, 0)+0.3, f'{v}\u2192{vr}', ha='center', fontsize=9)
axes[1].set_title('Exemplo: valores antes e depois da ReLU', fontsize=13)
axes[1].legend()
axes[1].axhline(0, color='k', linewidth=1)

plt.tight_layout()
plt.show()

---
## Pipeline Completo: Convolução + Bias + ReLU

A fórmula completa de **uma camada de CNN**:

$$\boxed{Im_{L+1}[i,j] = \max\left(0,\ b + \sum_{(m,n) \in \omega} Im_L[i+m, j+n] \cdot K[m,n]\right)}$$

Ou seja: **ReLU( Convolução + Bias )**

> **Resumindo o pipeline:** A imagem entra → o kernel extrai uma característica (ex: bordas) → o bias faz um ajuste fino → a ReLU descarta o que não é relevante (negativos viram zero). O resultado é uma imagem que destaca apenas as características importantes.

In [ ]:
# Pipeline completo na imagem da aula
bias_val = -0.1  # valor do bias (ajuste fino)

# Etapa 1: Convolução - desliza o kernel pela imagem extraindo características
step1 = convolucao_manual(img, K, padding=1)

# Etapa 2: Soma o bias a TODOS os pixels (desloca valores para cima ou para baixo)
step2 = step1 + bias_val

# Etapa 3: ReLU - tudo que ficou negativo vira zero (descarta o irrelevante)
step3 = relu(step2)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, data, titulo in [
    (axes[0], img, '$Im_L$ (Entrada)'),
    (axes[1], step1, '1. Convolu\u00e7\u00e3o'),
    (axes[2], step2, f'2. + Bias ({bias_val})'),
    (axes[3], step3, '3. ReLU (sa\u00edda final)'),
]:
    im = ax.imshow(data, cmap='gray')
    ax.set_title(titulo, fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Pipeline CNN: Entrada \u2192 Convolu\u00e7\u00e3o \u2192 +Bias \u2192 ReLU \u2192 Sa\u00edda',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Ap\u00f3s a ReLU: valores negativos viraram preto (zero).')
print('S\u00f3 as bordas mais significativas permaneceram!')

In [ ]:
# Efeito do Bias combinado com ReLU
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
biases = [0.0, -0.1, -0.2, -0.4]

for idx, b in enumerate(biases):
    sem_relu = conv_img + b
    com_relu = relu(sem_relu)

    axes[0, idx].imshow(sem_relu, cmap='gray')
    axes[0, idx].set_title(f'Conv + Bias={b:+.1f}', fontsize=12)

    axes[1, idx].imshow(com_relu, cmap='gray')
    pct = 100 * np.mean(com_relu == 0)
    axes[1, idx].set_title(f'+ ReLU ({pct:.0f}% descartado)', fontsize=12)

axes[0, 0].set_ylabel('Sem ReLU', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Com ReLU', fontsize=14, fontweight='bold')

plt.suptitle('Bias + ReLU: ajuste fino. Mais negativo = mais agressivo (descarta mais)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Bias=0: nenhum ajuste')
print('Bias=-0.1: ajuste suave (ideal para come\u00e7ar)')
print('Bias=-0.4: for\u00e7a demais \u2192 perde informa\u00e7\u00e3o importante!')

---
## Diferentes Kernels (Slide da Aula)

O professor mostrou 3 convoluções diferentes na mesma imagem:

| Kernel | Tamanho | O que faz | Resultado |
|--------|---------|-----------|----------|
| Detector de bordas | 3x3 | Encontra mudanças de intensidade | Contornos dos objetos |
| Blur (borramento) | 9x9 | Média dos vizinhos | Imagem suavizada/borrada |
| Identidade | 3x3 | 1 no centro, 0 no resto | Mantém a imagem original |

> **Por que kernels diferentes?** Na CNN, cada camada usa **vários kernels** simultaneamente. Um detecta bordas verticais, outro bordas horizontais, outro texturas. Cada kernel gera um **canal de saída** diferente. Assim, a imagem de entrada com 1 canal pode se transformar em 16, 64 ou 128 canais — cada um com uma característica extraída.

> Para explorar visualmente diferentes kernels, acesse: **[Simulador Interativo de Kernels](https://setosa.io/ev/image-kernels/)** (link compartilhado pelo colega Tiago na aula)

In [ ]:
kernels = {
    'Bordas Verticais (3x3)': (np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=float), 1),
    'Blur (9x9)': (np.ones((9,9), dtype=float) / 81.0, 4),
    'Identidade (3x3)': (np.array([[0,0,0],[0,1,0],[0,0,0]], dtype=float), 1),
}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('$Im_L$ (Entrada)', fontsize=14)

for idx, (nome, (kernel, pad)) in enumerate(kernels.items()):
    resultado = convolucao_manual(img, kernel, padding=pad)
    axes[idx+1].imshow(resultado, cmap='gray')
    axes[idx+1].set_title(nome, fontsize=12)

plt.suptitle('Diferentes kernels = diferentes caracter\u00edsticas extra\u00eddas da mesma imagem',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('Bordas: destaca contornos (cap\u00f4, telhado, cal\u00e7ada)')
print('Blur: suaviza tudo (como uma foto fora de foco)')
print('Identidade: n\u00e3o muda nada (multiplica por 1 no centro)')

---
## Canais RGB e Profundidade da CNN

Na CNN real, a cada camada o número de canais **aumenta**:

```
Entrada: 3 canais (RGB)
  \u2193 Camada 1: 16 filtros \u2192 16 canais
  \u2193 Camada 2: 64 filtros \u2192 64 canais
  \u2193 Camada 3: 128 filtros \u2192 128 canais
  \u2193 ...
  \u2193 Flatten \u2192 vetor \u2192 camadas densas \u2192 predição
```

Cada canal extraí características cada vez mais abstratas: das bordas simples até formas complexas como olhos, boca, sorriso.

In [ ]:
# Simular: 1 canal de entrada -> 3 canais de saída (3 kernels diferentes)
k_bordas_v = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=float)
k_bordas_h = np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=float)
k_blur = np.ones((3,3), dtype=float) / 9.0

canal1 = relu(convolucao_manual(img, k_bordas_v, padding=1) - 0.1)
canal2 = relu(convolucao_manual(img, k_bordas_h, padding=1) - 0.1)
canal3 = convolucao_manual(img, k_blur, padding=1)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Entrada\n(1 canal)', fontsize=13)

for ax, canal, titulo in [
    (axes[1], canal1, 'Canal 1\nBordas Verticais'),
    (axes[2], canal2, 'Canal 2\nBordas Horizontais'),
    (axes[3], canal3, 'Canal 3\nBlur'),
]:
    ax.imshow(canal, cmap='gray')
    ax.set_title(titulo, fontsize=12)

plt.suptitle('1 canal \u2192 3 canais (3 filtros diferentes) | Na CNN real: 3\u219216\u219264\u2192128',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Implementação com PyTorch

Na prática, não precisamos implementar a convolução manualmente. O **PyTorch** já tem tudo pronto! A camada `nn.Conv2d` faz convolução + bias automaticamente:

```python
conv = nn.Conv2d(in_channels=1,   # 1 canal de entrada (cinza)
                 out_channels=1,  # 1 canal de saída
                 kernel_size=3)   # kernel 3x3
```

> **O que é PyTorch?** É uma biblioteca de programação (em Python) usada para criar e treinar modelos de IA. Ela já tem pronta toda a lógica de convolução, bias, ReLU e treinamento. É como ter uma caixa de ferramentas completa.

In [ ]:
# Importar PyTorch - biblioteca que já tem convolução, bias e ReLU prontos
import torch
import torch.nn as nn           # nn = neural networks (camadas prontas)
import torch.nn.functional as F  # F = funções como ReLU

# Criar uma camada convolucional com PyTorch
# in_channels=1  -> entrada: 1 canal (escala de cinza)
# out_channels=1 -> saída: 1 canal
# kernel_size=3  -> kernel 3x3
# padding=1      -> manter resolução
# bias=True      -> incluir bias
conv_layer = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, padding=1, bias=True)

# Atribuir nosso kernel e bias manualmente (para comparar com o resultado manual)
# torch.no_grad() = desativa o cálculo de gradientes (não estamos treinando agora)
with torch.no_grad():
    conv_layer.weight.copy_(torch.tensor([[[[-1.,0.,1.],[-1.,0.,1.],[-1.,0.,1.]]]]))
    conv_layer.bias.copy_(torch.tensor([-0.1]))

# Converter imagem numpy para tensor PyTorch
# PyTorch espera formato: (batch, canais, altura, largura)
# unsqueeze(0) adiciona uma dimensão -> primeiro para batch, segundo para canal
img_t = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
print(f'Tensor de entrada: {img_t.shape}  (1 imagem, 1 canal, {img.shape[0]}x{img.shape[1]})')

# Aplicar o pipeline completo em UMA LINHA:
# conv_layer() = convolução + bias (automático)
# F.relu()     = função ReLU
with torch.no_grad():
    out = F.relu(conv_layer(img_t))

print(f'Tensor de sa\u00edda:   {out.shape}')
print('\nCom PyTorch, o pipeline inteiro cabe em 1 linha!')

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Entrada', fontsize=14)
axes[1].imshow(out.squeeze().numpy(), cmap='gray')
axes[1].set_title('PyTorch: Conv + Bias + ReLU', fontsize=14)
plt.suptitle('Implementa\u00e7\u00e3o com PyTorch (nn.Conv2d)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Múltiplos filtros de uma só vez: 1 canal de entrada -> 3 canais de saída
# Isso simula o que acontece numa CNN real: cada filtro extrai uma característica diferente
conv_multi = nn.Conv2d(1, 3, 3, padding=1, bias=False)  # 1 entrada, 3 saídas, kernel 3x3

# Definir os 3 filtros manualmente
with torch.no_grad():
    conv_multi.weight.copy_(torch.tensor([
        [[[-1.,0.,1.],[-1.,0.,1.],[-1.,0.,1.]]],
        [[[-1.,-1.,-1.],[0.,0.,0.],[1.,1.,1.]]],
        [[[1/9]*3,[1/9]*3,[1/9]*3]]
    ]))

with torch.no_grad():
    saida = F.relu(conv_multi(img_t))

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
nomes = ['Entrada (1 canal)', 'Canal 1: Bordas V', 'Canal 2: Bordas H', 'Canal 3: Blur']
axes[0].imshow(img, cmap='gray')
axes[0].set_title(nomes[0], fontsize=13)
for c in range(3):
    axes[c+1].imshow(saida[0,c].numpy(), cmap='gray')
    axes[c+1].set_title(nomes[c+1], fontsize=13)

plt.suptitle(f'PyTorch: 1 canal \u2192 3 canais | {img_t.shape} \u2192 {saida.shape}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Experimente você mesmo!

Coloque sua própria imagem na pasta `images/` e rode o código abaixo.

In [ ]:
# TROQUE AQUI pelo caminho da sua imagem:
MINHA_IMAGEM = 'images/imagem_aerea_professor.png'

# Escolha o kernel:
meu_kernel = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=float)

# Escolha o bias:
meu_bias = -0.1

# Pipeline
minha_img = np.array(Image.open(MINHA_IMAGEM).convert('L')).astype(float) / 255.0
conv_r = convolucao_manual(minha_img, meu_kernel, padding=1)
relu_r = relu(conv_r + meu_bias)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, data, titulo in [
    (axes[0], minha_img, 'Entrada'),
    (axes[1], conv_r + meu_bias, f'Conv + Bias ({meu_bias})'),
    (axes[2], relu_r, 'ReLU (sa\u00edda)'),
]:
    ax.imshow(data, cmap='gray')
    ax.set_title(titulo, fontsize=13)
plt.suptitle('Sua imagem no pipeline: Conv \u2192 Bias \u2192 ReLU', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Resumo da Aula

| Conceito | O que é | Para que serve |
|----------|---------|---------------|
| **Convolução** | Produto interno entre janela e kernel | Extrair características (bordas, textura) |
| **Kernel** | Pequena matriz de valores escolhidos | Define QUAL característica extrair |
| **Padding** | Preencher bordas com zeros | Manter resolução da imagem |
| **Stride** | Tamanho do passo da janela | Controlar resolução de saída |
| **Bias** | Número somado após convolução | Ajuste fino da extração |
| **ReLU** | max(0, z) | Não-linearidade (essencial para aprender) |

### Pipeline de cada camada CNN:
$$\boxed{Im_{L+1} = ReLU\left(Convolu\text{\u00e7\u00e3o}(Im_L, K) + bias\right)}$$

### Próximos passos:
- Aprofundar bias + ReLU com mais exemplos
- Camadas densas (Fully Connected)
- Gradiente descendente e Backpropagation
- Espaço latente

### Links úteis:
- [Simulador Visual de Kernels](https://setosa.io/ev/image-kernels/) (compartilhado pelo Tiago na aula)
- [Documentação PyTorch Conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)